In [ ]:
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET

# Paths
base_path = Path("/kaggle/input/datasets/uditag21/idddetection/IDD_Detection")
working_path = Path("/kaggle/working/idd_yolo")

(working_path / "labels/train").mkdir(parents=True, exist_ok=True)
(working_path / "labels/val").mkdir(parents=True, exist_ok=True)

annotations_path = base_path / "Annotations"

classes = [
    "person",
    "rider",
    "car",
    "truck",
    "bus",
    "motorcycle",
    "bicycle",
    "autorickshaw",
    "animal",
    "traffic light",
    "traffic sign"
]

def convert_annotation(xml_file, output_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    size = root.find("size")
    w = int(size.find("width").text)
    h = int(size.find("height").text)

    lines_to_write = []

    for obj in root.iter("object"):
        cls = obj.find("name").text

        if cls not in classes:
            continue

        cls_id = classes.index(cls)

        xmlbox = obj.find("bndbox")
        xmin = float(xmlbox.find("xmin").text)
        xmax = float(xmlbox.find("xmax").text)
        ymin = float(xmlbox.find("ymin").text)
        ymax = float(xmlbox.find("ymax").text)

        # Convert to YOLO format
        x_center = ((xmin + xmax) / 2.0) / w
        y_center = ((ymin + ymax) / 2.0) / h
        width = (xmax - xmin) / w
        height = (ymax - ymin) / h

        lines_to_write.append(
            f"{cls_id} {x_center} {y_center} {width} {height}"
        )

    # Only create label file if at least one valid object exists
    if len(lines_to_write) > 0:
        with open(output_file, "w") as f:
            f.write("\n".join(lines_to_write))


def process_split(split_file, split_name):
    with open(base_path / split_file) as f:
        lines = f.read().strip().split()

    for line in lines:
        xml_file = annotations_path / f"{line}.xml"
        label_dest = working_path / f"labels/{split_name}/{line}.txt"

        label_dest.parent.mkdir(parents=True, exist_ok=True)

        if xml_file.exists():
            convert_annotation(xml_file, label_dest)


process_split("train.txt", "train")
process_split("val.txt", "val")

print("Label conversion complete.")

Label conversion complete.


In [6]:
import os

# Create train/val image folders inside /working
train_dir = working_path / "images/train"
val_dir = working_path / "images/val"
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

def link_images(split_file, target_dir):
    with open(base_path / split_file) as f:
        lines = f.read().strip().split()
    for line in lines:
        # Each line may include subfolder info like "frontFar/BLR-.../001542_r"
        src = base_path / "JPEGImages" / f"{line}.jpg"
        dst = target_dir / f"{line}.jpg"

        # Ensure subfolders exist in target_dir
        dst.parent.mkdir(parents=True, exist_ok=True)

        if src.exists() and not dst.exists():
            os.symlink(src, dst)

# Link train and val images into /working/images
link_images("train.txt", train_dir)
link_images("val.txt", val_dir)

print("Symlinks created for train and val images in /working/images.")


Symlinks created for train and val images in /working/images.


In [7]:
yaml_content = """
path: /kaggle/working/idd_yolo

train: images/train
val: images/val

names:
"""

for i, cls in enumerate(classes):
    yaml_content += f"  {i}: {cls}\n"

with open("/kaggle/working/idd.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)


path: /kaggle/working/idd_yolo

train: images/train
val: images/val

names:
  0: person
  1: rider
  2: car
  3: truck
  4: bus
  5: motorcycle
  6: bicycle
  7: autorickshaw
  8: animal
  9: traffic light
  10: traffic sign



In [8]:
!ls /kaggle/working/idd_yolo/labels/train | head

frontFar
frontNear
highquality_16k
rearNear
sideLeft
sideRight


In [9]:
!ls /kaggle/input/datasets/uditag21/idddetection/IDD_Detection/JPEGImages/frontFar | head

BLR-2018-03-22_17-39-26_2_frontFar
BLR-2018-03-22_17-39-26_3_frontFar
BLR-2018-04-16_15-24-27_frontFar
BLR-2018-04-16_15-34-27_frontFar
BLR-2018-04-16_15-44-27_frontFar
BLR-2018-04-16_15-54-27_frontFar
BLR-2018-04-16_16-04-27_frontFar
BLR-2018-04-16_16-14-27_frontFar
BLR-2018-04-19_17-06-55_frontFar
BLR-2018-04-19_17-16-55_frontFar


In [ ]:
model = YOLO("yolov8m.pt")

model.train(
    data="/kaggle/working/idd.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    device=0,
    amp=True,
    workers=4,
    resume=False
)

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/idd.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo